In [1]:
# ==============================================
# NRAT SCRAPER FOR 2025
# ==============================================

!pip install requests beautifulsoup4 tqdm -q

import requests
from bs4 import BeautifulSoup
import os
import time
import re
from datetime import datetime, timedelta
from urllib.parse import urlencode
from tqdm import tqdm
import zipfile

# Import this with a different name to avoid conflict
from google.colab import files as colab_files

# ==============================================
# CONFIGURATION - 2025
# ==============================================

# Base search URL (without date parameters)
BASE_SEARCH_URL = "https://nrat.ukrintei.ua/searchdb/?"
BASE_PARAMS = {
    '_token': '6C0elymE1XyE8AOayLEsuYco3JewHUAF0DazKx9Y',
    'typeSearch2': 'ok',
    'typeCategory[]': '0',
    'lcSource': '',
    'authorSearch': '',
    'specialnistSearch[]': '0',
    'temaSearch2': '',
    'textSearch': '',
    'registrationNumberSearch': '',
    'firm_id': '0',
    'sortOrder': 'registration_date',
    'sortDir': 'desc',
    'tab': 'big'
}

# Date range settings for 2025
START_DATE = "2025-01-01"
END_DATE = "2025-12-31"
DAYS_PER_CHUNK = 1

# Download settings
DOWNLOAD_FOLDER = "nrat_pdfs_2025"
MAX_RESULTS_PER_CHUNK = 1000

# ==============================================
# SETUP
# ==============================================

print("🚀 Starting NRAT PDF Downloader for 2025")
print("=" * 70)
print(f"Date range: {START_DATE} to {END_DATE}")
print(f"Days per chunk: {DAYS_PER_CHUNK}")
print(f"Max results per chunk: {MAX_RESULTS_PER_CHUNK}")
print(f"Download folder: {DOWNLOAD_FOLDER}")
print("=" * 70)

# Create download folder
os.makedirs(DOWNLOAD_FOLDER, exist_ok=True)

# Setup session
session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.9,uk;q=0.8',
})

# ==============================================
# DATE RANGE FUNCTIONS
# ==============================================

def generate_date_ranges(start_date_str, end_date_str, days_per_chunk):
    """Generate 1-day date ranges between start and end dates"""
    start_date = datetime.strptime(start_date_str, "%Y-%m-%d")
    end_date = datetime.strptime(end_date_str, "%Y-%m-%d")

    date_ranges = []
    current_start = start_date

    while current_start <= end_date:
        current_end = min(current_start + timedelta(days=days_per_chunk - 1), end_date)
        date_ranges.append((
            current_start.strftime("%Y-%m-%d"),
            current_end.strftime("%Y-%m-%d")
        ))
        current_start = current_end + timedelta(days=1)

    return date_ranges

def build_search_url(date_from, date_to, page=1):
    """Build search URL with specific date range and page"""
    params = BASE_PARAMS.copy()
    params['dateFromSearch'] = date_from
    params['dateToSearch'] = date_to
    params['page'] = str(page)

    return BASE_SEARCH_URL + urlencode(params, doseq=True)

# ==============================================
# PAGINATION FUNCTIONS
# ==============================================

def extract_total_pages(soup):
    """Extract total number of pages from pagination"""
    try:
        pagination = soup.find('ul', class_='pagination')
        if pagination:
            page_links = pagination.find_all('a')
            page_numbers = []

            for link in page_links:
                try:
                    if link.text.strip().isdigit():
                        page_numbers.append(int(link.text.strip()))
                    elif link.get('href'):
                        href = link.get('href')
                        if 'page=' in href:
                            match = re.search(r'page=(\d+)', href)
                            if match:
                                page_numbers.append(int(match.group(1)))
                except:
                    continue

            if page_numbers:
                return max(page_numbers)

        page_info = soup.find('div', class_='page_info')
        if page_info:
            text = page_info.get_text(strip=True)
            match = re.search(r'Знайдено документів:\s*(\d+)', text)
            if match:
                total_results = int(match.group(1))
                total_pages = (total_results + 9) // 10
                return min(total_pages, 85)

        return 1

    except Exception as e:
        print(f"  ⚠️ Error extracting pagination: {e}")
        return 1

def extract_total_results(soup):
    """Extract total number of results"""
    try:
        page_info = soup.find('div', class_='page_info')
        if page_info:
            text = page_info.get_text(strip=True)
            match = re.search(r'Знайдено документів:\s*(\d+)', text)
            if match:
                return int(match.group(1))
    except:
        pass
    return 0

# ==============================================
# CORE SCRAPING FUNCTIONS
# ==============================================

def get_search_results_page(url):
    """Extract search results from a single page"""
    try:
        response = session.get(url, timeout=30)
        response.raise_for_status()

        soup = BeautifulSoup(response.content, 'html.parser')
        results = []

        result_cards = soup.find_all('div', class_='my-card-body')

        for card in result_cards:
            link_tag = card.find('a', target='_blank')

            if link_tag and link_tag.get('href'):
                reg_number = link_tag.get_text(strip=True)
                detail_url = link_tag['href']

                card_text = card.get_text(separator=' ', strip=True)
                title_parts = card_text.split('Керівник:')
                if len(title_parts) > 1:
                    title = title_parts[0].strip()
                else:
                    title = card_text[:100].strip()

                url_parts = detail_url.split('/')
                doc_id = url_parts[-1] if url_parts[-1] else url_parts[-2]
                pdf_url = f"https://dir.ukrintei.ua/view/ok/{doc_id}"

                results.append({
                    'registration': reg_number,
                    'detail_url': detail_url,
                    'pdf_url': pdf_url,
                    'title': title,
                    'doc_id': doc_id
                })

        return results, soup

    except Exception as e:
        print(f"❌ Error fetching search page: {str(e)}")
        return [], None

def download_pdf(pdf_url, registration, title, folder, retry_count=2):
    """Download PDF file with retry logic"""

    safe_reg = re.sub(r'[^\w\-_.]', '_', registration)
    safe_title = re.sub(r'[^\w\-_. ]', '_', title[:50]) if title else ""

    if safe_title:
        filename = f"{safe_reg}_{safe_title}.pdf"
    else:
        filename = f"{safe_reg}.pdf"

    filename = re.sub(r'_+', '_', filename)
    filepath = os.path.join(folder, filename)

    if os.path.exists(filepath):
        print(f"    ⚠️ File exists, skipping: {filename}")
        return False, filepath

    for attempt in range(retry_count):
        try:
            print(f"    📥 Downloading (attempt {attempt + 1}/{retry_count})...")

            response = session.get(pdf_url, stream=True, timeout=60)
            response.raise_for_status()

            file_size = int(response.headers.get('content-length', 0))

            with open(filepath, 'wb') as f:
                if file_size > 0:
                    with tqdm(
                        desc="      Progress",
                        total=file_size,
                        unit='B',
                        unit_scale=True,
                        unit_divisor=1024,
                        miniters=1,
                        leave=False
                    ) as pbar:
                        for chunk in response.iter_content(chunk_size=8192):
                            if chunk:
                                f.write(chunk)
                                pbar.update(len(chunk))
                else:
                    for chunk in response.iter_content(chunk_size=8192):
                        if chunk:
                            f.write(chunk)

            if os.path.exists(filepath):
                actual_size = os.path.getsize(filepath)

                if actual_size > 0:
                    with open(filepath, 'rb') as f:
                        first_bytes = f.read(4)
                        if first_bytes.startswith(b'%PDF'):
                            size_kb = actual_size / 1024
                            print(f"    ✅ Saved: {filename} ({size_kb:.1f} KB)")
                            return True, filepath
                        else:
                            os.remove(filepath)
                            print(f"    ❌ Downloaded file is not a PDF")
                            return False, None
                else:
                    print(f"    ❌ Downloaded empty file")
                    return False, None

        except Exception as e:
            print(f"    ❌ Download failed: {str(e)}")
            if attempt < retry_count - 1:
                print(f"    ⏳ Retrying in 3 seconds...")
                time.sleep(3)

    return False, None

# ==============================================
# MAIN SCRAPING LOGIC
# ==============================================

def scrape_date_range(date_from, date_to):
    """Scrape all results for a specific date range"""
    print(f"\n📅 Processing date range: {date_from} to {date_to}")
    print("-" * 50)

    page = 1
    all_results = []
    total_pages = None

    while True:
        url = build_search_url(date_from, date_to, page)
        print(f"  Page {page}: {url[:80]}...")

        results, soup = get_search_results_page(url)

        if not results and page == 1:
            print(f"  ⚠️ No results found for this date range")
            break

        if soup and page == 1:
            total_pages = extract_total_pages(soup)
            total_results = extract_total_results(soup)
            print(f"  Found {total_results} results, {total_pages} pages")

            if total_results >= 1000:
                print(f"  ⚠️ WARNING: Hit 1000+ result limit for this date range")
                print(f"  Consider using smaller date chunks")

        if results:
            all_results.extend(results)
            print(f"  Extracted {len(results)} results from page {page}")

            process_results_page(results, date_from, date_to, page)

        if total_pages is None:
            if soup:
                total_pages = extract_total_pages(soup)

        if total_pages and page >= total_pages:
            print(f"  ✅ Finished all {total_pages} pages for this date range")
            break

        if not results:
            print(f"  ⚠️ No results on page {page}, stopping")
            break

        page += 1

        if page <= total_pages:
            delay = 3
            print(f"  ⏳ Waiting {delay} seconds before next page...")
            time.sleep(delay)

    return all_results

def process_results_page(results, date_from, date_to, page):
    """Process and download PDFs from a results page"""

    month_folder = datetime.strptime(date_from, "%Y-%m-%d").strftime("%Y-%m")
    date_folder = os.path.join(DOWNLOAD_FOLDER, month_folder, f"{date_from}")
    os.makedirs(date_folder, exist_ok=True)

    page_folder = os.path.join(date_folder, f"page_{page}")
    os.makedirs(page_folder, exist_ok=True)

    print(f"  Downloading to: {page_folder}")

    successful = 0
    failed = 0

    for i, result in enumerate(results, 1):
        print(f"\n    [{i}/{len(results)}] {result['registration']}")
        print(f"       Title: {result['title'][:60]}...")

        success, filepath = download_pdf(
            result['pdf_url'],
            result['registration'],
            result['title'],
            page_folder
        )

        if success:
            successful += 1
        else:
            failed += 1

        if i < len(results):
            time.sleep(1)

    print(f"\n    📊 Page {page} summary: {successful} successful, {failed} failed")

    summary_file = os.path.join(page_folder, "summary.txt")
    with open(summary_file, 'w', encoding='utf-8') as f:
        f.write(f"Date range: {date_from} to {date_to}\n")
        f.write(f"Page: {page}\n")
        f.write(f"Total results: {len(results)}\n")
        f.write(f"Successful downloads: {successful}\n")
        f.write(f"Failed downloads: {failed}\n")
        f.write(f"Download time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")

        for result in results:
            f.write(f"{result['registration']}: {result['title'][:80]}...\n")

# ==============================================
# MAIN EXECUTION
# ==============================================

def main():
    """Main scraping function"""
    print("\n" + "=" * 70)
    print("GENERATING DATE RANGES FOR 2025")
    print("=" * 70)

    date_ranges = generate_date_ranges(START_DATE, END_DATE, DAYS_PER_CHUNK)
    print(f"Generated {len(date_ranges)} date chunks (1 day each)")

    for i, (date_from, date_to) in enumerate(date_ranges, 1):
        print(f"\nChunk {i}/{len(date_ranges)}: {date_from} to {date_to}")

        scrape_date_range(date_from, date_to)

        if i < len(date_ranges):
            delay = 5
            print(f"\n⏳ Finished chunk {i}, waiting {delay} seconds before next chunk...")
            time.sleep(delay)

    print("\n" + "=" * 70)
    print("2025 SCRAPING COMPLETE!")
    print("=" * 70)

    create_final_summary()

def create_final_summary():
    """Create final summary report"""
    print("\n📊 GENERATING FINAL SUMMARY FOR 2025")

    summary_file = os.path.join(DOWNLOAD_FOLDER, "FINAL_SUMMARY_2025.txt")

    total_pdfs = 0
    total_folders = 0

    with open(summary_file, 'w', encoding='utf-8') as f:
        f.write("=" * 60 + "\n")
        f.write("NRAT PDF DOWNLOAD - 2025 FINAL SUMMARY\n")
        f.write("=" * 60 + "\n\n")
        f.write(f"Date range: {START_DATE} to {END_DATE}\n")
        f.write(f"Days per chunk: {DAYS_PER_CHUNK}\n")
        f.write(f"Download completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")

        for root, dirs, filenames in os.walk(DOWNLOAD_FOLDER):
            if root != DOWNLOAD_FOLDER:
                rel_path = os.path.relpath(root, DOWNLOAD_FOLDER)
                pdf_files = [file for file in filenames if file.endswith('.pdf')]

                if pdf_files:
                    f.write(f"\n📁 {rel_path}/\n")
                    f.write(f"   PDF files: {len(pdf_files)}\n")

                    total_folders += 1
                    total_pdfs += len(pdf_files)

                    for pdf in pdf_files[:5]:
                        size = os.path.getsize(os.path.join(root, pdf)) / 1024
                        f.write(f"   • {pdf} ({size:.1f} KB)\n")

                    if len(pdf_files) > 5:
                        f.write(f"   ... and {len(pdf_files) - 5} more\n")

        f.write(f"\n" + "=" * 60 + "\n")
        f.write(f"TOTALS:\n")
        f.write(f"  Total date chunks processed: {len(date_ranges)}\n")
        f.write(f"  Total folders with PDFs: {total_folders}\n")
        f.write(f"  Total PDF files downloaded: {total_pdfs}\n")

    print(f"✅ Final summary saved: {summary_file}")
    print(f"📊 Total PDFs downloaded: {total_pdfs}")

    print("\n📄 ALL DOWNLOADED PDFS:")
    pdf_files_list = []
    for root, dirs, filenames in os.walk(DOWNLOAD_FOLDER):
        for file in filenames:
            if file.endswith('.pdf'):
                pdf_files_list.append(os.path.join(root, file))

    if pdf_files_list:
        print(f"Found {len(pdf_files_list)} PDF files in total")

        for i, pdf in enumerate(pdf_files_list[:10], 1):
            size = os.path.getsize(pdf) / 1024
            rel_path = os.path.relpath(pdf, DOWNLOAD_FOLDER)
            print(f"  {i}. {rel_path} ({size:.1f} KB)")

        if len(pdf_files_list) > 10:
            print(f"  ... and {len(pdf_files_list) - 10} more")
    else:
        print("  No PDF files found")

# ==============================================
# DOWNLOAD UTILITIES
# ==============================================

def create_zip_archive():
    """Create ZIP archive of all downloaded files"""
    print("\n🗜️ Creating ZIP archive for 2025...")

    zip_filename = f"nrat_2025_pdfs_{datetime.now().strftime('%Y%m%d_%H%M%S')}.zip"

    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, filenames in os.walk(DOWNLOAD_FOLDER):
            for file in filenames:
                filepath = os.path.join(root, file)
                arcname = os.path.relpath(filepath, DOWNLOAD_FOLDER)
                zipf.write(filepath, arcname)

    zip_size = os.path.getsize(zip_filename) / (1024 * 1024)
    print(f"✅ ZIP created: {zip_filename} ({zip_size:.2f} MB)")

    return zip_filename

def save_to_drive():
    """Save all files to Google Drive"""
    try:
        from google.colab import drive
        drive.mount('/content/drive')

        drive_folder = "/content/drive/MyDrive/nrat_pdfs_2025"
        os.makedirs(drive_folder, exist_ok=True)

        import shutil
        shutil.copytree(DOWNLOAD_FOLDER, drive_folder, dirs_exist_ok=True)

        print(f"✅ All files saved to Google Drive: {drive_folder}")

    except Exception as e:
        print(f"❌ Error saving to Google Drive: {str(e)}")

# ==============================================
# RUN THE SCRAPER
# ==============================================

if __name__ == "__main__":
    date_ranges = generate_date_ranges(START_DATE, END_DATE, DAYS_PER_CHUNK)

    main()

    print("\n" + "=" * 70)
    print("AUTOMATIC DOWNLOAD INITIATED FOR 2025")
    print("=" * 70)

    pdf_files_list = []
    for root, dirs, filenames in os.walk(DOWNLOAD_FOLDER):
        for file in filenames:
            if file.endswith('.pdf'):
                pdf_files_list.append(os.path.join(root, file))

    total_pdfs = len(pdf_files_list)

    if total_pdfs > 0:
        zip_file = create_zip_archive()
        print(f"\n📦 Total PDFs downloaded for 2025: {total_pdfs}")
        print("⏳ Downloading ZIP file automatically...")
        colab_files.download(zip_file)
        print("✅ ZIP file download initiated!")
    else:
        print("❌ No PDFs were downloaded for 2025, so no zip file to download.")

    print("\n" + "=" * 70)
    print("✅ 2025 PROCESS COMPLETE!")
    print("=" * 70)

    print("\nAdditional options:")
    print("1. To save files to Google Drive, run: save_to_drive()")
    print("2. To re-download the zip file, run: colab_files.download(zip_file)")
    print("3. To access individual files, use the Files sidebar in Colab")

Streaming output truncated to the last 5000 lines.
    [4/10] 0225U004348
       Title: Head: Kalatur Kateryna A. . Establishing the peculiarities o...
    📥 Downloading (attempt 1/2)...
    ✅ Saved: 0225U004348_Head_ Kalatur Kateryna A. . Establishing the pecul.pdf (261.0 KB)

    [5/10] 0225U004346
       Title: Head: Hanna M. Dydyk-Meuch . Historical memory of Ukrainians...
    📥 Downloading (attempt 1/2)...
    ✅ Saved: 0225U004346_Head_ Hanna M. Dydyk-Meuch . Historical memory of .pdf (260.5 KB)

    [6/10] 0225U004345
       Title: Head: Hypalo Vira D. . Between East and West: the funeral cu...
    📥 Downloading (attempt 1/2)...
    ✅ Saved: 0225U004345_Head_ Hypalo Vira D. . Between East and West_ the .pdf (260.4 KB)

    [7/10] 0225U004343
       Title: Head: Yarmoluk Serhii M. . Investigation of Mycobacterium tu...
    📥 Downloading (attempt 1/2)...
    ✅ Saved: 0225U004343_Head_ Yarmoluk Serhii M. . Investigation of Mycoba.pdf (260.8 KB)

    [8/10] 0225U004342
       Title: 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ ZIP file download initiated!

✅ 2025 PROCESS COMPLETE!

Additional options:
1. To save files to Google Drive, run: save_to_drive()
2. To re-download the zip file, run: colab_files.download(zip_file)
3. To access individual files, use the Files sidebar in Colab
